In [13]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# ===== INICIAR SPARK =====
spark = SparkSession.builder \
    .appName("Analisis_MercadoLibre_S7") \
    .config("spark.mongodb.read.connection.uri", "mongodb://bigdata_mongodb:27017/proyecto_bigdata.smartphones_ml")\
    .config("spark.jars.packages", "org.mongodb.spark:mongo-spark-connector_2.12:10.1.1") \
    .getOrCreate()

print(" Spark iniciado correctamente\n")

# ===== CARGAR DATOS =====
df = spark.read.format("mongodb").load()

print(" ESTRUCTURA DE DATOS:")
df.printSchema()

print("\n MUESTRA DE DATOS:")
df.select("titulo", "precio_actual", "descuento", "timestamp").show(5, truncate=False)

# ===== LIMPIEZA DE DATOS =====
print("\n LIMPIEZA DE DATOS:")
print(f"Registros originales: {df.count()}")

# Filtrar registros válidos
df_limpio = df.filter(
    (F.col("titulo").isNotNull()) & 
    (F.col("precio_actual") > 0) &
    (F.col("grupo") == "E-Commerce_iPhones")
)

print(f"Registros válidos: {df_limpio.count()}")

# ===== ANÁLISIS POR RANGO DE PRECIOS =====
print("\n ANÁLISIS DE PRECIOS:")

df_limpio.groupBy("descuento") \
    .agg(
        F.count("titulo").alias("cantidad"),
        F.round(F.avg("precio_actual"), 0).alias("precio_promedio"),
        F.min("precio_actual").alias("precio_minimo"),
        F.max("precio_actual").alias("precio_maximo")
    ) \
    .orderBy("descuento", ascending=False) \
    .show()

# ===== PRODUCTOS CON MAYOR DESCUENTO =====
print("\n TOP 5 MAYORES DESCUENTOS:")

df_limpio.filter(F.col("descuento") == True) \
    .select("titulo", "precio_actual", "precio_original", "porcentaje_descuento") \
    .orderBy(F.col("porcentaje_descuento").desc()) \
    .show(5, truncate=False)

print("\n Análisis completado")

 Spark iniciado correctamente

 ESTRUCTURA DE DATOS:
root
 |-- _id: string (nullable = true)
 |-- descuento: boolean (nullable = true)
 |-- descuento_porcentaje: integer (nullable = true)
 |-- fecha_scraping: string (nullable = true)
 |-- grupo: string (nullable = true)
 |-- imagen_url: string (nullable = true)
 |-- marca: string (nullable = true)
 |-- metodo_captura: string (nullable = true)
 |-- pagina: integer (nullable = true)
 |-- porcentaje_descuento: double (nullable = true)
 |-- precio_actual: integer (nullable = true)
 |-- precio_original: double (nullable = true)
 |-- producto_id: string (nullable = true)
 |-- responsable: string (nullable = true)
 |-- tiene_descuento: boolean (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- titulo: string (nullable = true)
 |-- url: string (nullable = true)


 MUESTRA DE DATOS:
+------------------------------------------------------------+-------------+---------+-----------------------+
|titulo                              